# Compute durations and ages

The arithmetic looks simple — "subtract two dates" — but there are a few traps. Months and years aren't fixed durations, business days exclude weekends, and "age" is calendar-based not duration-based. This recipe covers the common cases and what to reach for in each.

## Elapsed time between two datetimes

Straight subtraction gives a `timedelta`. For consistent units, use `.total_seconds()` and divide.

In [ ]:
from datetime import datetime, timedelta

start = datetime(2026, 4, 21, 9, 0)
end   = datetime(2026, 4, 22, 15, 30)

gap = end - start
print(gap)                                    # 1 day, 6:30:00
print(gap.total_seconds(), "seconds")
print(gap.total_seconds() / 3600, "hours")
print(gap.days, "whole days")

`gap.days` is the whole-days component — not the total elapsed days. `timedelta(hours=23)` has `.days == 0`. For total days as a float, divide: `gap.total_seconds() / 86400`.

## Age in years

"How old is someone born on 2000-06-15, as of today?" — the intuitive answer is a whole number, which a `timedelta` can't give you directly. The clean approach is calendar-based:

In [ ]:
from datetime import date

def age_in_years(dob: date, as_of: date) -> int:
    years = as_of.year - dob.year
    # Subtract one if their birthday hasn't happened yet this year
    if (as_of.month, as_of.day) < (dob.month, dob.day):
        years -= 1
    return years

print(age_in_years(date(2000, 6, 15), date(2026, 4, 21)))   # 25
print(age_in_years(date(2000, 6, 15), date(2026, 6, 15)))   # 26
print(age_in_years(date(2000, 6, 15), date(2026, 6, 14)))   # 25

Using `(as_of - dob).days / 365.25` is an approximation — fine for "average age of cohort" aggregates, wrong if you need the integer number of birthdays someone has had.

## "N months from now" with `relativedelta`

`timedelta` doesn't do months (a month isn't a fixed number of days). For calendar-based arithmetic — "the first of next month", "in three months", "one year from today" — use `dateutil.relativedelta`.

In [ ]:
# Uncomment if dateutil is installed:
# from dateutil.relativedelta import relativedelta
# 
# today = date(2026, 4, 21)
# print(today + relativedelta(months=3))     # 2026-07-21
# print(today + relativedelta(years=1))      # 2027-04-21
# print(today + relativedelta(months=1, day=1))   # 2026-05-01 (first of next month)
# 
# # End-of-month behaviour: adding a month to the 31st clamps to the target month's last day
# print(date(2026, 1, 31) + relativedelta(months=1))   # 2026-02-28

# Without dateutil, a pure-stdlib approach for 'N months from today':
def add_months(d: date, n: int) -> date:
    total = d.month - 1 + n
    year = d.year + total // 12
    month = total % 12 + 1
    # Clamp day to last day of target month
    from calendar import monthrange
    day = min(d.day, monthrange(year, month)[1])
    return date(year, month, day)

print(add_months(date(2026, 4, 21), 3))       # 2026-07-21
print(add_months(date(2026, 1, 31), 1))       # 2026-02-28

## Business days

If "days" means "working days" (excluding weekends and holidays), neither `timedelta` nor `relativedelta` helps. Use `numpy.busday_count`, or iterate manually.

In [ ]:
def business_days_between(start: date, end: date) -> int:
    """Number of weekdays strictly between start and end (inclusive of start, exclusive of end)."""
    from datetime import timedelta
    count = 0
    d = start
    while d < end:
        if d.weekday() < 5:     # Mon-Fri are 0-4
            count += 1
        d += timedelta(days=1)
    return count

# Tue 2026-04-21 to Tue 2026-04-28 — 5 business days
print(business_days_between(date(2026, 4, 21), date(2026, 4, 28)))

For anything beyond weekends — bank holidays, shortened Fridays, country-specific rules — reach for the `holidays` package or `pandas.tseries.offsets.BusinessDay` with a custom `calendar` argument.

## Formatting a duration for humans

`timedelta` prints as `1 day, 6:30:00` — fine for logs, ugly for a UI. A small helper:

In [ ]:
def humanise(td: timedelta) -> str:
    total = int(td.total_seconds())
    days, rem = divmod(total, 86400)
    hours, rem = divmod(rem, 3600)
    minutes, _ = divmod(rem, 60)
    parts = []
    if days:    parts.append(f"{days}d")
    if hours:   parts.append(f"{hours}h")
    if minutes: parts.append(f"{minutes}m")
    return " ".join(parts) or "0m"

print(humanise(timedelta(days=1, hours=6, minutes=30)))
print(humanise(timedelta(minutes=45)))
print(humanise(timedelta(0)))

## Tips

- `timedelta` for fixed durations (seconds, hours, days). `relativedelta` for calendar durations (months, years).
- "Age" is calendar-based: subtract years, adjust for whether the birthday has happened yet.
- `.total_seconds()` is your escape hatch when you need a single numeric value.
- For business days and holidays, don't reinvent — use `numpy.busday_count` or `holidays`.